<a href="https://colab.research.google.com/github/Tefong35/TefongEugene/blob/main/Kundenverlauf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
- Person- und Business-Account-Daten abgleicht
- Migrationen erkennt (Zuwanderung, Abwanderung)
- Ergebnisse auf 3 Excel-Blätter verteilt:
  - Blatt 0: Übersicht mit Bestandskunden + Zuwanderung – Abwanderung
  - Blatt 1: Zuwanderung (neu hinzugekommene Kunden)
  - Blatt 2: Abwanderung (nicht mehr vorhandene Kunden)
- Täglich ausführbar ist

*Python-Skript:*
```python
import pandas as pd
from datetime import datetime
import openpyxl

1. Daten abrufen (simuliert durch CSV oder direkte DB-Abfrage ersetzen)
Beispiel: alte Daten (Bestand vom Vortag)
mapping_df = pd.read_excel("Mapping_Table.xlsx", sheet_name="1. Blatt")

Neue Daten abrufen (aktuelle Person- und Business-Accounts z. B. aus DB)
Achtung: Die Datenbankabfrage muss angepasst werden!
Beispiel:
new_df = pd.read_sql("SELECT * FROM person_accounts", connection)

Hier als Platzhalter: neue Daten einlesen (aktuell)
new_df = pd.read_excel("Aktuelle_Daten.xlsx")  # aktuelle Datenquelle

2. Vergleich anhand der Kundennummer (angenommen Spaltenname = 'Kundennummer')
alte_kunden = set(mapping_df['Kundennummer'])
neue_kunden = set(new_df['Kundennummer'])

3. Zuwanderung: Kunden, die neu sind
zuwanderung = neue_kunden - alte_kunden
05.08.25, 21:50 - ChatGPT: df_zuwanderung = new_df[new_df['Kundennummer'].isin(zuwanderung)]

4. Abwanderung: Kunden, die verschwunden sind
abwanderung = alte_kunden - neue_kunden
df_abwanderung = mapping_df[mapping_df['Kundennummer'].isin(abwanderung)]

5. Bestand: Kunden, die geblieben sind
bestand = neue_kunden & alte_kunden
df_bestand = new_df[new_df['Kundennummer'].isin(bestand)]

6. Gesamttabelle mit Bestand + Zuwanderung - Abwanderung (für Übersicht)
gesamt_df = pd.concat([df_bestand, df_zuwanderung])

7. Ergebnisse in neue Excel-Datei schreiben
output_file = "Migration_Analyse.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    gesamt_df.to_excel(writer, sheet_name="0. Übersicht", index=False)
    df_zuwanderung.to_excel(writer, sheet_name="1. Zuwanderung", index=False)
    df_abwanderung.to_excel(writer, sheet_name="2. Abwanderung", index=False)

print(f"Analyse abgeschlossen: Datei gespeichert unter {output_file}")
```

---

Hinweise:
- Ersetze `Aktuelle_Daten.xlsx` durch deine DB-Abfrage für aktuelle Daten.
- `Kundennummer` sollte der Primärschlüssel sein.
- Automatisiere das Skript mit einem Scheduler (z. B. `cron` oder Windows-Taskplaner).

Möchtest du zusätzlich ein Logging oder tägliches Archivieren der Excel-Dateien?
05.08.25, 21:56 - Edmond: Das Matching von Person- und Business-account-Daten ausgib eine Tabelle z.B. Blatt 00, die als Basis-Tabelle zum Zeitpunkt heute dient. Ab morgen werden dann die weitere Blätte: Blatt 0, Blatt 1 und Blatt 2 ausgeführt.
05.08.25, 21:57 - ChatGPT: Perfekt! Basierend auf deinem Workflow – *heute wird die Basistabelle erstellt* (`Blatt 00`) und *ab morgen täglich Zuwanderung, Abwanderung, Bestand ermittelt* – hier eine passende Lösung.

✅ Zielstruktur:

- *Blatt "00. Basis"* → Heute erzeugt (Erstzustand)
- *Ab morgen:*
  - *Blatt "0. Übersicht"* → Bestand + Zuwanderung – Abwanderung
  - *Blatt "1. Zuwanderung"*
  - *Blatt "2. Abwanderung"*

---

✅ Schritt 1: Heutige Erstellung der *Basistabelle* (`00. Basis`)
```python
import pandas as pd
from datetime import datetime

Neue Daten (heute)
Beispiel: neue Daten abrufen (aus Datenbank)
person_df = pd.read_excel("Person_accounts.xlsx")
business_df = pd.read_excel("Business_accounts.xlsx")

Merge auf gemeinsame ID (z. B. Kundennummer)
basis_df = pd.merge(person_df, business_df, on="Kundennummer", how="outer")

Basis-Tabelle für heute erstellen
output_file = "Mapping_Table.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    basis_df.to_excel(writer, sheet_name="00. Basis", index=False)

print("Basistabelle erstellt: Mapping_Table.xlsx")
```

---

✅ Schritt 2: Ab morgen – Vergleich mit der Basistabelle
```python
import pandas as pd
from datetime import datetime

Aktuelle Daten (z. B. täglicher Export oder Datenbankabfrage)
05.08.25, 21:57 - ChatGPT: neue_df = pd.read_excel("Aktuelle_Daten.xlsx")

Basisdaten (gespeichert in Blatt "00. Basis")
basis_df = pd.read_excel("Mapping_Table.xlsx", sheet_name="00. Basis")

Kunden-IDs
alte_kunden = set(basis_df["Kundennummer"])
neue_kunden = set(neue_df["Kundennummer"])

Zuwanderung
zuwanderung = neue_kunden - alte_kunden
df_zuwanderung = neue_df[neue_df["Kundennummer"].isin(zuwanderung)]

Abwanderung
abwanderung = alte_kunden - neue_kunden
df_abwanderung = basis_df[basis_df["Kundennummer"].isin(abwanderung)]

Bestand
bestand = neue_kunden & alte_kunden
df_bestand = neue_df[neue_df["Kundennummer"].isin(bestand)]

Übersicht
gesamt_df = pd.concat([df_bestand, df_zuwanderung])

Ergebnisse speichern
output_file = "Mapping_Table.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl", mode='a', if_sheet_exists='replace') as writer:
    gesamt_df.to_excel(writer, sheet_name="0. Übersicht", index=False)
    df_zuwanderung.to_excel(writer, sheet_name="1. Zuwanderung", index=False)
    df_abwanderung.to_excel(writer, sheet_name="2. Abwanderung", index=False)

print("Tägliche Analyse abgeschlossen.")
```

---

💡Zusatzoption:
- Automatisieren per *Taskplaner*
- Archiv der täglichen Ergebnisse (z. B. mit Datumsstempel)
05.08.25, 21:57 - ChatGPT: Möchtest du auch farbliche Formatierung oder ein Verlaufsdiagramm mitentwickeln?
05.08.25, 22:25 - Edmond: Ich möchte automatisieren und täglich durführen (aktualisieren), auch farbliche Formatierung und Verlaufsdiagramm mitentwickeln.
05.08.25, 22:25 - ChatGPT: Hier ist ein *vollständiges Python-Skript*, das deine tägliche Analyse automatisiert, eine Excel-Datei mit:

- *Zuwanderung* (Blatt 1),
- *Abwanderung* (Blatt 2),
- *Bestand mit Zuwanderung* (Blatt 0),
- *Verlaufsdiagramm* (Bestand über Tage),
- *Farbliche Formatierung* (z. B. grün für Zuwanderung, rot für Abwanderung)

✅ Voraussetzungen:
- `openpyxl`, `pandas`, `matplotlib`, `datetime`
- Basisdatei existiert: `Mapping_Table.xlsx` mit „00. Basis“
- Tägliche Datenquelle als `Aktuelle_Daten.xlsx`

---

✅ Skript zur täglichen Ausführung
```python
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from datetime import datetime

=== EINSTELLUNGEN ===
basis_file = "Mapping_Table.xlsx"
heute_datum = datetime.today().strftime("%Y-%m-%d")
aktuell_file = "Aktuelle_Daten.xlsx"

=== 1. Lese Basisdaten & aktuelle Daten ===
basis_df = pd.read_excel(basis_file, sheet_name="00. Basis")
aktuell_df = pd.read_excel(aktuell_file)

=== 2. Extrahiere Kundennummern ===
alte_kunden = set(basis_df["Kundennummer"])
neue_kunden = set(aktuell_df["Kundennummer"])

zuwanderung = neue_kunden - alte_kunden
abwanderung = alte_kunden - neue_kunden
bestand = neue_kunden & alte_kunden
05.08.25, 22:26 - ChatGPT: df_zuwanderung = aktuell_df[aktuell_df["Kundennummer"].isin(zuwanderung)]
df_abwanderung = basis_df[basis_df["Kundennummer"].isin(abwanderung)]
df_bestand = aktuell_df[aktuell_df["Kundennummer"].isin(bestand)]
gesamt_df = pd.concat([df_bestand, df_zuwanderung])

=== 3. Verlaufsdaten aktualisieren ===
try:
    verlauf_df = pd.read_excel(basis_file, sheet_name="Verlauf", index_col=0)
except:
    verlauf_df = pd.DataFrame()

verlauf_df.loc[heute_datum, "Bestand"] = len(bestand)
verlauf_df.loc[heute_datum, "Zuwanderung"] = len(zuwanderung)
verlauf_df.loc[heute_datum, "Abwanderung"] = len(abwanderung)

=== 4. Speichern in Excel ===
with pd.ExcelWriter(basis_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    gesamt_df.to_excel(writer, sheet_name="0. Übersicht", index=False)
    df_zuwanderung.to_excel(writer, sheet_name="1. Zuwanderung", index=False)
    df_abwanderung.to_excel(writer, sheet_name="2. Abwanderung", index=False)
    verlauf_df.to_excel(writer, sheet_name="Verlauf")

=== 5. Farbliche Formatierung ===
wb = load_workbook(basis_file)
ws = wb["1. Zuwanderung"]
fill_green = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
05.08.25, 22:26 - ChatGPT: for cell in row:
        cell.fill = fill_green

ws2 = wb["2. Abwanderung"]
fill_red = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row, min_col=1, max_col=ws2.max_column):
    for cell in row:
        cell.fill = fill_red

wb.save(basis_file)

=== 6. Diagramm Verlauf ===
plt.figure(figsize=(10, 5))
verlauf_df.plot(marker="o")
plt.title("Täglicher Verlauf: Bestand, Zuwanderung, Abwanderung")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("Kundenverlauf.png")
```

---

📊 Ergebnis:
- *Excel-Datei* `Mapping_Table.xlsx` mit farbigen Blättern:
  - *0. Übersicht*: Bestand + neue Kunden
  - *1. Zuwanderung* (grün)
  - *2. Abwanderung* (rot)
  - *Verlauf*: Historie
- *PNG-Datei* `Kundenverlauf.png` mit Liniendiagramm

---

🛠 Automatisierung
Nutze *Windows Aufgabenplanung* oder `cron` (Linux/Mac), um das Skript täglich auszuführen.

Möchtest du ein Setup dafür?
05.08.25, 22:30 - Edmond: Ich möchte ein Setup
05.08.25, 22:31 - ChatGPT: Perfekt! Hier ist eine einfache Anleitung für ein *automatisiertes Setup*, um dein tägliches Python-Skript zur Kunden-Migration (Zuwanderung/Abwanderung) *täglich automatisch* auszuführen.

---

✅ Voraussetzungen

1. Dein Skript liegt z. B. unter:
   `C:\Projekte\kundenmonitor\tagesupdate.py`

2. Du hast *Python korrekt installiert* (z. B. in `C:\Python39\python.exe`)

---

🪟 Setup unter *Windows* (mit Aufgabenplanung)

1. Aufgabenplanung öffnen
- Windows-Taste → *„Aufgabenplanung“* suchen und öffnen

2. Neue Aufgabe erstellen
- Klick auf *„Aufgabe erstellen“*
- Name: `Tägliches Kundenupdate`
- Haken setzen bei *„Mit höchsten Privilegien ausführen“*

3. Tab „Trigger“ → „Neu“
- Beginnen: „Täglich“
- Startdatum & Uhrzeit z. B. `08:00 Uhr`
- OK

4. Tab „Aktionen“ → „Neu“
- Aktion: *„Programm starten“*
- Programm/Skript: Pfad zu deiner `python.exe`, z. B.:
  `C:\Python39\python.exe`
- Argumente hinzufügen:
  Pfad zu deinem Skript, z. B.:
  `C:\Projekte\kundenmonitor\tagesupdate.py`
- Starten in: (optional) Pfad zum Ordner, z. B.:
  `C:\Projekte\kundenmonitor\`
- OK

5. Tab „Bedingungen“ & „Einstellungen“
- Du kannst z. B. „Nur ausführen, wenn mit Strom verbunden“ deaktivieren
- OK drücken

---

🧪 Test
- Rechtsklick → *„Ausführen“*
- Überprüfe ob:
05.08.25, 22:31 - ChatGPT: - `Mapping_Table.xlsx` aktualisiert wurde
  - `Kundenverlauf.png` erstellt wurde

---

📝 Optional: Logging hinzufügen
Füge ins Skript z. B. am Anfang ein:
```python
with open("log.txt", "a") as log:
    log.write(f"{datetime.today()} – Update ausgeführt\n")
```

---

Wenn du auf macOS oder Linux arbeitest, kann ich dir den Cronjob dafür erklären. Möchtest du das?
05.08.25, 22:34 - Edmond: Kundenverlauf.png wird täglich aktualisiert oder führe jeder Tag eine Grafik ?
05.08.25, 22:34 - ChatGPT: Gute Frage!

Du hast *zwei Möglichkeiten*, wie du mit `Kundenverlauf.png` umgehen kannst:

---

*1. Grafik täglich überschreiben (eine Datei)*
- Die Datei `Kundenverlauf.png` wird *jeden Tag aktualisiert*, ersetzt die alte Version.
- Vorteil: Nur *eine Datei*, immer *aktuell*.
- Code:
  ```python
  plt.savefig("Kundenverlauf.png")
  ```

---

*2. Tägliche Version abspeichern (eine pro Tag)*
- Jeden Tag wird *eine neue Grafik gespeichert*, z. B.:
  `Kundenverlauf_2025-08-05.png`, `Kundenverlauf_2025-08-06.png`
- Vorteil: Du kannst *Rückblick machen*, Tagesvergleich, Reporting etc.
- Code:
  ```python
  from datetime import datetime
  today = datetime.today().strftime("%Y-%m-%d")
  plt.savefig(f"Kundenverlauf_{today}.png")
  ```

---

*Empfehlung:*
Wenn du langfristig Entwicklungen vergleichen möchtest → Option 2.
Wenn du nur das aktuelle Bild brauchst → Option 1.

Soll ich dir beides kombinieren, z. B. *immer aktualisieren UND zusätzlich täglich speichern*?
05.08.25, 22:41 - Edmond: Nur aktualisieren reicht estmal, jedoch brauche ich ein Skript, um langfristig Entwicklungen zu vergleichen. Z.B. monatlich Kundenverlauf.png
05.08.25, 22:41 - ChatGPT: Alles klar! Du möchtest:

- *Täglich* die Datei `Kundenverlauf.png` *aktualisieren* (wie bisher).
- *Zusätzlich einmal pro Monat* eine *Version speichern*, z. B. `Kundenverlauf_2025-08.png`, um Entwicklungen langfristig zu vergleichen.

Hier ist ein *Kombinations-Skript*:

```python
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

Beispiel-Daten: ersetze durch echte Kundenverläufe
df = pd.read_csv("kundenverlauf.csv")  # z. B. mit Spalten: Datum, Bestand

Plot erstellen
plt.figure(figsize=(10, 6))
plt.plot(df["Datum"], df["Bestand"], marker='o', color='steelblue')
plt.title("Kundenverlauf")
plt.xlabel("Datum")
plt.ylabel("Kundenanzahl")
plt.grid(True)

1. Täglich aktualisierte Datei
plt.savefig("Kundenverlauf.png")

2. Monatlich speichern (am 1. Tag des Monats)
today = datetime.today()
if today.day == 1:
    monat = today.strftime("%Y-%m")
    plt.savefig(f"Kundenverlauf_{monat}.png")

plt.close()
```

*Hinweis:*
- Du brauchst eine Datei wie `kundenverlauf.csv` oder ziehst die Daten direkt aus deiner DB.
- Falls du das Ganze automatisierst (z. B. via `cron` oder `Task Scheduler`), läuft das Skript täglich.

Möchtest du den Verlauf auch als Excel-Datei sichern?
05.08.25, 22:42 - Edmond: Ja
05.08.25, 22:42 - ChatGPT: Perfekt – hier ist das *erweiterte Skript*, das täglich:

1. *Kundenverlauf.png* aktualisiert
2. *Kundenverlauf_YYYY-MM.png* (monatlich) speichert
3. *Excel-Datei Kundenverlauf.xlsx* aktualisiert mit Diagramm
4. (Optional) Diagramm als Bild in Excel einfügt

```python
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import openpyxl
from openpyxl.chart import LineChart, Reference
from openpyxl.styles import Font
import os

=== Daten laden ===
df = pd.read_csv("kundenverlauf.csv")  # Spalten: Datum, Bestand
df["Datum"] = pd.to_datetime(df["Datum"])

=== PNG-Grafik ===
plt.figure(figsize=(10, 6))
plt.plot(df["Datum"], df["Bestand"], marker="o", color="steelblue")
plt.title("Kundenverlauf")
plt.xlabel("Datum")
plt.ylabel("Kundenanzahl")
plt.grid(True)

Speichern: täglich
plt.savefig("Kundenverlauf.png")

Speichern: monatlich
today = datetime.today()
if today.day == 1:
    monat = today.strftime("%Y-%m")
    plt.savefig(f"Kundenverlauf_{monat}.png")
plt.close()

=== Excel-Datei ===
excel_file = "Kundenverlauf.xlsx"
df.to_excel(excel_file, index=False, sheet_name="Verlauf")

Diagramm in Excel einfügen
wb = openpyxl.load_workbook(excel_file)
ws = wb["Verlauf"]

Linie zeichnen
chart = LineChart()
chart.title = "Kundenverlauf"
chart.style = 13
05.08.25, 22:42 - ChatGPT: chart.y_axis.title = "Kundenanzahl"
chart.x_axis.title = "Datum"

data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
ws.add_chart(chart, "D2")

Speichern
wb.save(excel_file)
```

*Strukturierte Dateiübersicht:*
- `kundenverlauf.csv` → täglicher Input
- `Kundenverlauf.png` → tägliche Grafik
- `Kundenverlauf_YYYY-MM.png` → monatliche Archivierung
- `Kundenverlauf.xlsx` → Übersicht & Verlauf mit Diagramm

Möchtest du das Skript auch automatisch täglich ausführen (Windows/Unix)?
05.08.25, 22:46 - Edmond: Bitte berechne Anhang der Zuwanderung, Auswarung, bestand etc den Kundenverlauf.
05.08.25, 22:46 - ChatGPT: Gerne! Hier ist ein *Python-Skript*, das basierend auf den täglichen Daten für:

- *Zuwanderung* (neue Kunden),
- *Abwanderung* (verlorene Kunden) und
- *Bestand* (bestehende Kunden)

den täglichen *Kundenverlauf* berechnet und aktualisiert.

---

*Voraussetzung:*
Du hast täglich drei Dateien (oder Tabellen):
- `bestand.csv` → bestehende Kunden heute
- `zuwanderung.csv` → neue Kunden heute
- `abwanderung.csv` → verlorene Kunden heute

---

✅ *Python-Skript zur Berechnung des Kundenverlaufs:*

```python
import pandas as pd
from datetime import datetime

Aktuelles Datum
today = datetime.today().strftime("%Y-%m-%d")

Lade tägliche Daten
bestand = pd.read_csv("bestand.csv")         # z. B. Spalte: kundennummer
zuwanderung = pd.read_csv("zuwanderung.csv") # neue Kunden
abwanderung = pd.read_csv("abwanderung.csv") # verlorene Kunden

Berechne Kennzahlen
anzahl_bestand = len(bestand)
anzahl_zuwanderung = len(zuwanderung)
anzahl_abwanderung = len(abwanderung)

Bestand Vortag (falls Verlauf schon vorhanden ist)
try:
    verlauf = pd.read_csv("kundenverlauf.csv")
    bestand_gestern = verlauf.iloc[-1]["Bestand"]
except FileNotFoundError:
    verlauf = pd.DataFrame(columns=["Datum", "Bestand", "Zuwanderung", "Abwanderung"])
    bestand_gestern = 0
05.08.25, 22:46 - ChatGPT: Aktueller Bestand berechnet (alternativ: aus bestand.csv direkt nehmen)
aktueller_bestand = bestand_gestern + anzahl_zuwanderung - anzahl_abwanderung

Neue Zeile anhängen
neuer_tag = pd.DataFrame([{
    "Datum": today,
    "Bestand": aktueller_bestand,
    "Zuwanderung": anzahl_zuwanderung,
    "Abwanderung": anzahl_abwanderung
}])

verlauf = pd.concat([verlauf, neuer_tag], ignore_index=True)

Speichern
verlauf.to_csv("kundenverlauf.csv", index=False)
```

---

🔄 Ergebnis:
- *kundenverlauf.csv* enthält den Verlauf:
  - Spalten: `Datum`, `Bestand`, `Zuwanderung`, `Abwanderung`

Du kannst jetzt mit dem Kundenverlauf weiterarbeiten und z. B. ein Diagramm erzeugen.

Möchtest du zusätzlich eine Visualisierung (z. B. Linien- oder Balkendiagramm)?
05.08.25, 22:49 - Edmond: Ja
05.08.25, 22:50 - ChatGPT: Hier ist der ergänzte *Python-Code* zur täglichen *Visualisierung des Kundenverlaufs* auf Basis der Datei `kundenverlauf.csv`. Die Grafik wird als *PNG-Datei* gespeichert und automatisch *aktualisiert*.

---

📊 *Visualisierung des Kundenverlaufs*

```python
import pandas as pd
import matplotlib.pyplot as plt

Lade Kundenverlauf
verlauf = pd.read_csv("kundenverlauf.csv", parse_dates=["Datum"])

Plot erstellen
plt.figure(figsize=(12, 6))
plt.plot(verlauf["Datum"], verlauf["Bestand"], label="Bestand", color="blue", marker="o")
plt.bar(verlauf["Datum"], verlauf["Zuwanderung"], label="Zuwanderung", color="green", alpha=0.5)
plt.bar(verlauf["Datum"], -verlauf["Abwanderung"], label="Abwanderung", color="red", alpha=0.5)

Formatierung
plt.title("Kundenverlauf")
plt.xlabel("Datum")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()

Grafik speichern
plt.savefig("kundenverlauf.png")
plt.close()
```

---

🗂️ Ergebnis:
- `kundenverlauf.png`: enthält die Entwicklung von Bestand, Zuwanderung und Abwanderung
- Automatisch täglich aktualisierbar, z. B. per Zeitplan oder Batch

Möchtest du zusätzlich eine monatliche Aggregation für Vergleich oder Trenddarstellung?
05.08.25, 22:50 - Edmond: Ja
05.08.25, 22:50 - ChatGPT: Hier ist eine *Erweiterung des Skripts* zur *monatlichen Aggregation des Kundenverlaufs* – ideal, um *langfristige Entwicklungen zu vergleichen*. Dabei wird eine neue Grafik `kundenverlauf_monatlich.png` erzeugt:

---

📦 *Monatliche Kundenverlauf-Analyse in Python*

```python
import pandas as pd
import matplotlib.pyplot as plt

Daten laden
df = pd.read_csv("kundenverlauf.csv", parse_dates=["Datum"])

Monat extrahieren
df["Monat"] = df["Datum"].dt.to_period("M").astype(str)

Monatlich aggregieren
monatlich = df.groupby("Monat").agg({
    "Bestand": "last",
    "Zuwanderung": "sum",
    "Abwanderung": "sum"
}).reset_index()

Plot erstellen
plt.figure(figsize=(12, 6))
plt.plot(monatlich["Monat"], monatlich["Bestand"], label="Bestand", marker="o", color="blue")
plt.bar(monatlich["Monat"], monatlich["Zuwanderung"], label="Zuwanderung", color="green", alpha=0.6)
plt.bar(monatlich["Monat"], -monatlich["Abwanderung"], label="Abwanderung", color="red", alpha=0.6)

Formatierung
plt.title("Monatlicher Kundenverlauf")
plt.xlabel("Monat")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()

Grafik speichern
plt.savefig("kundenverlauf_monatlich.png")
plt.close()
```

---

🖼️ Ergebnis:
05.08.25, 22:50 - ChatGPT: - `kundenverlauf_monatlich.png`: zeigt Kundenentwicklung über Monate
- Vergleich pro Monat von Bestand, Zuwanderung, Abwanderung

Möchtest du zusätzlich einen gleitenden Durchschnitt oder eine Trendlinie einbauen?
05.08.25, 22:51 - Edmond: Ja
05.08.25, 22:51 - ChatGPT: Hier ist die erweiterte Version mit *gleitendem Durchschnitt (Moving Average)* und einer *Trendlinie* für den Kundenbestand. Das hilft dir, saisonale Effekte und langfristige Entwicklungen zu erkennen:

---

📊 *Python-Skript mit gleitendem Durchschnitt & Trendlinie*

```python
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

CSV laden
df = pd.read_csv("kundenverlauf.csv", parse_dates=["Datum"])
df["Monat"] = df["Datum"].dt.to_period("M").astype(str)

Monatlich aggregieren
monatlich = df.groupby("Monat").agg({
    "Bestand": "last",
    "Zuwanderung": "sum",
    "Abwanderung": "sum"
}).reset_index()

Gleitender Durchschnitt (z. B. über 3 Monate)
monatlich["Bestand_MA"] = monatlich["Bestand"].rolling(window=3, min_periods=1).mean()

Trendlinie berechnen (lineare Regression)
x = np.arange(len(monatlich))
z = np.polyfit(x, monatlich["Bestand"], 1)
trend = np.poly1d(z)(x)

Plot
plt.figure(figsize=(12, 6))
plt.plot(monatlich["Monat"], monatlich["Bestand"], label="Bestand", marker="o", color="blue")
plt.plot(monatlich["Monat"], monatlich["Bestand_MA"], label="Gleitender Durchschnitt", linestyle="--", color="orange")
plt.plot(monatlich["Monat"], trend, label="Trendlinie", linestyle=":", color="black")
05.08.25, 22:52 - ChatGPT: plt.bar(monatlich["Monat"], monatlich["Zuwanderung"], label="Zuwanderung", color="green", alpha=0.5)
plt.bar(monatlich["Monat"], -monatlich["Abwanderung"], label="Abwanderung", color="red", alpha=0.5)

plt.title("Kundenverlauf mit Trendanalyse")
plt.xlabel("Monat")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("kundenverlauf_monatlich_trend.png")
plt.close()
```

---

📝 Hinweise:
- `Bestand_MA`: Gleitender Durchschnitt über 3 Monate
- `Trendlinie`: zeigt langfristige Entwicklung (lineare Regression)
- Grafik wird gespeichert als *"kundenverlauf_monatlich_trend.png"*

Möchtest du das auch automatisieren – z. B. täglich oder monatlich generieren lassen?
05.08.25, 22:52 - Edmond: Monatlich
05.08.25, 22:53 - ChatGPT: Perfekt. Um den *Kundenverlauf mit Trendanalyse monatlich automatisch zu aktualisieren*, kannst du folgendes Setup nutzen:

---

✅ *1. Ordnerstruktur vorbereiten*
```plaintext
projektordner/
├── daten/
│   └── kundenverlauf.csv         # enthält täglich aktualisierte Kundendaten
├── ausgabe/
│   └── kundenverlauf_monatlich_trend.png
├── kundenverlauf_monatlich.py   # dein Python-Skript
```

---

✅ *2. Python-Skript für monatliches Update (`kundenverlauf_monatlich.py`)*

```python
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import datetime

Datenpfade
DATENPFAD = "daten/kundenverlauf.csv"
AUSGABEPFAD = "ausgabe/kundenverlauf_monatlich_trend.png"

Daten laden
df = pd.read_csv(DATENPFAD, parse_dates=["Datum"])
df["Monat"] = df["Datum"].dt.to_period("M").astype(str)

Monatlich aggregieren
monatlich = df.groupby("Monat").agg({
    "Bestand": "last",
    "Zuwanderung": "sum",
    "Abwanderung": "sum"
}).reset_index()

Gleitender Durchschnitt (3 Monate)
monatlich["Bestand_MA"] = monatlich["Bestand"].rolling(window=3, min_periods=1).mean()

Trendlinie (lineare Regression)
x = np.arange(len(monatlich))
z = np.polyfit(x, monatlich["Bestand"], 1)
trend = np.poly1d(z)(x)

Plot
plt.figure(figsize=(12, 6))
05.08.25, 22:53 - ChatGPT: plt.plot(monatlich["Monat"], monatlich["Bestand"], label="Bestand", marker="o", color="blue")
plt.plot(monatlich["Monat"], monatlich["Bestand_MA"], label="Gleitender Durchschnitt", linestyle="--", color="orange")
plt.plot(monatlich["Monat"], trend, label="Trendlinie", linestyle=":", color="black")
plt.bar(monatlich["Monat"], monatlich["Zuwanderung"], label="Zuwanderung", color="green", alpha=0.5)
plt.bar(monatlich["Monat"], -monatlich["Abwanderung"], label="Abwanderung", color="red", alpha=0.5)

plt.title("Kundenverlauf mit Trendanalyse (Monatlich)")
plt.xlabel("Monat")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()

Speichern
plt.savefig(AUSGABEPFAD)
plt.close()
```

---

✅ *3. Automatisierung (Monatlich ausführen)*

Option A: *Windows*
- Öffne „Aufgabenplanung“
- Trigger: „Monatlich“, z. B. jeder 1. um 7 Uhr
- Aktion:
  - `python "C:\Pfad\zu\kundenverlauf_monatlich.py"`

Option B: *Linux/macOS (Cronjob)*
Cron-Eintrag z. B. am 1. jeden Monats um 7 Uhr:

```bash
0 7 1 * * /usr/bin/python3 /pfad/zu/kundenverlauf_monatlich.py
```

---

Möchtest du zusätzlich automatisch die Grafik mit Datum archivieren (z. B. `kundenverlauf_2025-08.png`)?
05.08.25, 22:53 - ChatGPT: Unbegrenzte Nutzung freischalten 💬 Wusstest du, dass du Freunde zum Nachrichtenaustausch mit ChatGPT einladen und als Dankeschön 1 Tag lang unbegrenzt Foto-Uploads und Audionotizen nutzen kannst?
05.08.25, 22:54 - Edmond: Ja
05.08.25, 22:55 - ChatGPT: Super! Dann erweitern wir das Skript so, dass jeden Monat die Grafik *„Kundenverlauf_MM-YYYY.png“* gespeichert und archiviert wird. Zusätzlich bleibt die aktuelle Version *„kundenverlauf_monatlich_trend.png“* erhalten.

---

✅ *Erweiterte Version des Skripts*

```python
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import datetime

Datenpfade
DATENPFAD = "daten/kundenverlauf.csv"
AUSGABEORDNER = "ausgabe/"
HEUTE = datetime.today()
DATE_SUFFIX = HEUTE.strftime("%Y-%m")
ARCHIVDATEI = f"{AUSGABEORDNER}kundenverlauf_{DATE_SUFFIX}.png"
AKTUELLE_DATEI = f"{AUSGABEORDNER}kundenverlauf_monatlich_trend.png"

Daten laden
df = pd.read_csv(DATENPFAD, parse_dates=["Datum"])
df["Monat"] = df["Datum"].dt.to_period("M").astype(str)

Monatlich aggregieren
monatlich = df.groupby("Monat").agg({
    "Bestand": "last",
    "Zuwanderung": "sum",
    "Abwanderung": "sum"
}).reset_index()

Gleitender Durchschnitt und Trendlinie
monatlich["Bestand_MA"] = monatlich["Bestand"].rolling(window=3, min_periods=1).mean()
x = np.arange(len(monatlich))
trend = np.poly1d(np.polyfit(x, monatlich["Bestand"], 1))(x)

Plot
plt.figure(figsize=(12, 6))
plt.plot(monatlich["Monat"], monatlich["Bestand"], label="Bestand", marker="o", color="blue")
05.08.25, 22:55 - ChatGPT: plt.plot(monatlich["Monat"], monatlich["Bestand_MA"], label="Gleitender Durchschnitt", linestyle="--", color="orange")
plt.plot(monatlich["Monat"], trend, label="Trendlinie", linestyle=":", color="black")
plt.bar(monatlich["Monat"], monatlich["Zuwanderung"], label="Zuwanderung", color="green", alpha=0.5)
plt.bar(monatlich["Monat"], -monatlich["Abwanderung"], label="Abwanderung", color="red", alpha=0.5)

plt.title("Kundenverlauf mit Trendanalyse (Monatlich)")
plt.xlabel("Monat")
plt.ylabel("Anzahl Kunden")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()

Speichern
plt.savefig(ARCHIVDATEI)         # z. B. kundenverlauf_2025-08.png
plt.savefig(AKTUELLE_DATEI)      # immer aktuelle Version
plt.close()
```

---

✅ Ergebnis

Nach jeder Ausführung:
- *kundenverlauf_monatlich_trend.png* wird überschrieben
- *kundenverlauf_2025-08.png* wird archiviert

